In [1]:
import pandas as pd

Caracterización de las Fuentes de Datos

En esta sección se describen las fuentes de datos utilizadas en el prototipo del sistema de **Gestión Inteligente de Inventarios Farmacéuticos**. Las fuentes simulan los sistemas de información típicos presentes en una Institución Prestadora de Salud (IPS), donde la información relacionada con medicamentos suele encontrarse distribuida en múltiples sistemas operacionales.

Las fuentes consideradas corresponden a tres tipos de registros: dispensaciones de medicamentos desde el sistema de farmacia, compras de medicamentos desde el sistema ERP y registros históricos de consumo almacenados en hojas de cálculo. Cada fuente aporta información complementaria necesaria para calcular indicadores de inventario como el stock disponible, el consumo promedio y el punto de reorden.

**Fuente de Datos 1: Sistema de Dispensación de Farmacia**

La primera fuente de datos corresponde al registro de dispensaciones de medicamentos realizadas por el sistema de farmacia de la institución. Este sistema almacena las transacciones asociadas a la entrega de medicamentos a pacientes o a diferentes servicios clínicos dentro de la IPS.

Para efectos del prototipo, esta fuente se representa mediante el archivo **dispensaciones.csv**, el cual contiene registros individuales de dispensación.

Este tipo de datos tiene naturaleza transaccional, ya que cada registro corresponde a un evento específico de salida de inventario. Debido a la frecuencia con la que se dispensan medicamentos en un entorno hospitalario, esta fuente suele generar un volumen considerable de registros diarios.

Las variables típicas presentes en este tipo de fuente incluyen la fecha de dispensación, el código del medicamento, el nombre del medicamento, la cantidad dispensada, el servicio clínico al que se entregó el medicamento y el número de lote.

La información proveniente de esta fuente es fundamental para el pipeline ETL, ya que **permite calcular el consumo real de medicamentos** y, por tanto, representa el flujo de salida del inventario farmacéutico.

In [2]:
# Dataset de ejemplo generado en un .csv

dispensaciones = pd.DataFrame([
    ["D0001", "2025-02-01", "M001", "Acetaminofén 500 mg", "Tableta", "Urgencias", 20, "L202501", "tabletas"],
    ["D0002", "2025-02-01", "M002", "Amoxicilina 500 mg", "Cápsula", "Consulta Externa", 15, "L202502", "capsulas"],
    ["D0003", "2025-02-02", "M003", "Losartán 50 mg", "Tableta", "Hospitalización", 30, "L202503", "tabletas"],
    ["D0004", "2025-02-02", "M004", "Insulina NPH 100 UI/mL", "Frasco", "UCI", 5, "L202504", "frascos"],
    ["D0005", "2025-02-03", "M005", "Omeprazol 20 mg", "Cápsula", "Urgencias", 25, "L202505", "capsulas"],
    ["D0006", "2025-02-03", "M001", "Acetaminofén 500 mg", "Tableta", "Pediatría", 18, "L202501", "tabletas"],
    ["D0007", "2025-02-04", "M006", "Ceftriaxona 1 g", "Vial", "Hospitalización", 12, "L202506", "viales"],
    ["D0008", "2025-02-04", "M007", "Enoxaparina 40 mg", "Jeringa", "UCI", 8, "L202507", "jeringas"],
    ["D0009", "2025-02-05", "M003", "Losartán 50 mg", "Tableta", "Consulta Externa", 22, "L202503", "tabletas"],
    ["D0010", "2025-02-05", "M008", "Salbutamol inhalador", "Inhalador", "Urgencias", 6, "L202508", "inhaladores"]
], columns=[
    "id_dispensacion", "fecha_dispensacion", "codigo_medicamento", "nombre_medicamento",
    "presentacion", "servicio_clinico", "cantidad_dispensada", "lote", "unidad_medida"
]) # Columnas que se espera encontrar en un archivo de este tipo

dispensaciones.to_csv("dispensaciones.csv", index=False, encoding="utf-8-sig")
dispensaciones.head()

,id_dispensacion,fecha_dispensacion,codigo_medicamento,nombre_medicamento,presentacion,servicio_clinico,cantidad_dispensada,lote,unidad_medida
0,D0001,2025-02-01,M001,Acetaminofén 500 mg,Tableta,Urgencias,20,L202501,tabletas
1,D0002,2025-02-01,M002,Amoxicilina 500 mg,Cápsula,Consulta Externa,15,L202502,capsulas
2,D0003,2025-02-02,M003,Losartán 50 mg,Tableta,Hospitalización,30,L202503,tabletas
3,D0004,2025-02-02,M004,Insulina NPH 100 UI/mL,Frasco,UCI,5,L202504,frascos
4,D0005,2025-02-03,M005,Omeprazol 20 mg,Cápsula,Urgencias,25,L202505,capsulas


**Fuente de Datos 2: Sistema ERP de Compras**

La segunda fuente de datos corresponde al sistema de planificación de recursos empresariales (ERP) utilizado por la institución para gestionar el proceso de compras y abastecimiento de medicamentos.

En el prototipo, esta fuente se representa mediante el archivo **compras.csv**, el cual contiene registros asociados a las adquisiciones de medicamentos realizadas a proveedores.

Los datos contenidos en esta fuente describen el **flujo de entrada de inventario**, es decir, las cantidades de medicamentos que ingresan al sistema como resultado de órdenes de compra y procesos de abastecimiento.

Entre las variables típicas presentes en este tipo de registros se encuentran la fecha de compra, el proveedor, el código del medicamento, el nombre del medicamento, la cantidad adquirida y el costo unitario del producto.

Esta fuente es esencial para el análisis del inventario, ya que permite conocer los **ingresos de medicamentos al almacén farmacéutico** y calcular el balance entre las entradas y salidas de inventario.

In [3]:
compras = pd.DataFrame([
    ["C0001", "2025-01-20", "Drogas La Rebaja", "800123456", "M001", "Acetaminofén 500 mg", 500, 120.5, "L202501", "2027-01-15"],
    ["C0002", "2025-01-21", "Copservir", "900456789", "M002", "Amoxicilina 500 mg", 300, 450.0, "L202502", "2026-12-30"],
    ["C0003", "2025-01-22", "Cruz Verde", "901234567", "M003", "Losartán 50 mg", 400, 210.0, "L202503", "2027-02-10"],
    ["C0004", "2025-01-23", "Baxter Colombia", "860002222", "M004", "Insulina NPH 100 UI/mL", 100, 18500.0, "L202504", "2026-11-20"],
    ["C0005", "2025-01-24", "Copidrogas", "830111333", "M005", "Omeprazol 20 mg", 350, 310.0, "L202505", "2027-03-05"],
    ["C0006", "2025-01-25", "Pfizer Colombia", "860045678", "M006", "Ceftriaxona 1 g", 200, 3200.0, "L202506", "2026-10-18"],
    ["C0007", "2025-01-26", "Sanofi Aventis", "860007777", "M007", "Enoxaparina 40 mg", 150, 12500.0, "L202507", "2026-09-25"],
    ["C0008", "2025-01-27", "GSK Colombia", "860099999", "M008", "Salbutamol inhalador", 120, 9800.0, "L202508", "2027-01-01"]
], columns=[
    "id_compra", "fecha_compra", "proveedor", "nit_proveedor", "codigo_medicamento",
    "nombre_medicamento", "cantidad_comprada", "costo_unitario", "lote", "fecha_vencimiento"
])

compras.to_csv("compras.csv", index=False, encoding="utf-8-sig")
compras.head()

,id_compra,fecha_compra,proveedor,nit_proveedor,codigo_medicamento,nombre_medicamento,cantidad_comprada,costo_unitario,lote,fecha_vencimiento
0,C0001,2025-01-20,Drogas La Rebaja,800123456,M001,Acetaminofén 500 mg,500,120.5,L202501,2027-01-15
1,C0002,2025-01-21,Copservir,900456789,M002,Amoxicilina 500 mg,300,450.0,L202502,2026-12-30
2,C0003,2025-01-22,Cruz Verde,901234567,M003,Losartán 50 mg,400,210.0,L202503,2027-02-10
3,C0004,2025-01-23,Baxter Colombia,860002222,M004,Insulina NPH 100 UI/mL,100,18500.0,L202504,2026-11-20
4,C0005,2025-01-24,Copidrogas,830111333,M005,Omeprazol 20 mg,350,310.0,L202505,2027-03-05


**Fuente de Datos 3: Registro Histórico de Consumo**

La tercera fuente corresponde a registros históricos de consumo de medicamentos, los cuales suelen mantenerse en archivos de hojas de cálculo utilizados para análisis administrativos o reportes internos.

En el prototipo, esta información se representa mediante el archivo **consumos_historicos.xlsx**, el cual contiene datos consolidados de consumo de medicamentos en diferentes períodos de tiempo.

A diferencia de las fuentes anteriores, que contienen datos transaccionales detallados, esta fuente suele manejar datos agregados, por ejemplo consumo total por medicamento durante un período determinado.

Las variables típicas de esta fuente incluyen la fecha o período de registro, el código del medicamento, el nombre del medicamento, el consumo total registrado y el servicio clínico asociado.

La función principal de esta fuente dentro del pipeline ETL es proporcionar información histórica que permita calcular indicadores de demanda, como el **consumo promedio diario** y las tendencias de uso de los medicamentos.

In [4]:
consumos_historicos = pd.DataFrame([
    ["2024-11", "M001", "Acetaminofén 500 mg", "Urgencias", 620, 20.7, "tabletas"],
    ["2024-11", "M003", "Losartán 50 mg", "Hospitalización", 840, 28.0, "tabletas"],
    ["2024-11", "M004", "Insulina NPH 100 UI/mL", "UCI", 90, 3.0, "frascos"],
    ["2024-12", "M001", "Acetaminofén 500 mg", "Urgencias", 700, 22.6, "tabletas"],
    ["2024-12", "M005", "Omeprazol 20 mg", "Urgencias", 540, 17.4, "capsulas"],
    ["2024-12", "M006", "Ceftriaxona 1 g", "Hospitalización", 300, 9.7, "viales"],
    ["2025-01", "M003", "Losartán 50 mg", "Consulta Externa", 660, 21.3, "tabletas"],
    ["2025-01", "M007", "Enoxaparina 40 mg", "UCI", 180, 5.8, "jeringas"],
    ["2025-01", "M008", "Salbutamol inhalador", "Urgencias", 90, 2.9, "inhaladores"],
    ["2025-01", "M002", "Amoxicilina 500 mg", "Consulta Externa", 420, 13.5, "capsulas"]
], columns=[
    "periodo", "codigo_medicamento", "nombre_medicamento", "servicio_clinico",
    "consumo_total", "consumo_promedio_diario", "unidad_medida"
])

consumos_historicos.to_excel("consumos_historicos.xlsx", index=False)
consumos_historicos.head()

,periodo,codigo_medicamento,nombre_medicamento,servicio_clinico,consumo_total,consumo_promedio_diario,unidad_medida
0,2024-11,M001,Acetaminofén 500 mg,Urgencias,620,20.7,tabletas
1,2024-11,M003,Losartán 50 mg,Hospitalización,840,28.0,tabletas
2,2024-11,M004,Insulina NPH 100 UI/mL,UCI,90,3.0,frascos
3,2024-12,M001,Acetaminofén 500 mg,Urgencias,700,22.6,tabletas
4,2024-12,M005,Omeprazol 20 mg,Urgencias,540,17.4,capsulas


In [5]:
# 1. LECTURA DE LAS FUENTES
dispensaciones = pd.read_csv("dispensaciones.csv", encoding="utf-8-sig")
compras = pd.read_csv("compras.csv", encoding="utf-8-sig")
consumos_historicos = pd.read_excel("consumos_historicos.xlsx")

# 2. VISTA PREVIA
print("=== DISPENSACIONES ===")
display(dispensaciones.head())

print("=== COMPRAS ===")
display(compras.head())

print("=== CONSUMOS HISTÓRICOS ===")
display(consumos_historicos.head())

# 3. DIMENSIONES
print("Dimensiones de los datasets:")
print(f"dispensaciones: {dispensaciones.shape[0]} filas, {dispensaciones.shape[1]} columnas")
print(f"compras: {compras.shape[0]} filas, {compras.shape[1]} columnas")
print(f"consumos_historicos: {consumos_historicos.shape[0]} filas, {consumos_historicos.shape[1]} columnas")

# 4. TIPOS DE DATOS
print("\nTipos de datos - dispensaciones")
print(dispensaciones.dtypes)

print("\nTipos de datos - compras")
print(compras.dtypes)

print("\nTipos de datos - consumos_historicos")
print(consumos_historicos.dtypes)

# 5. VALORES NULOS
print("\nValores nulos por columna - dispensaciones")
print(dispensaciones.isnull().sum())

print("\nValores nulos por columna - compras")
print(compras.isnull().sum())

print("\nValores nulos por columna - consumos_historicos")
print(consumos_historicos.isnull().sum())

# 6. DUPLICADOS
print("\nRegistros duplicados:")
print(f"dispensaciones: {dispensaciones.duplicated().sum()}")
print(f"compras: {compras.duplicated().sum()}")
print(f"consumos_historicos: {consumos_historicos.duplicated().sum()}")

# 7. CONVERSIÓN BÁSICA DE FECHAS
dispensaciones["fecha_dispensacion"] = pd.to_datetime(dispensaciones["fecha_dispensacion"], errors="coerce")
compras["fecha_compra"] = pd.to_datetime(compras["fecha_compra"], errors="coerce")
compras["fecha_vencimiento"] = pd.to_datetime(compras["fecha_vencimiento"], errors="coerce")
consumos_historicos["periodo"] = pd.to_datetime(consumos_historicos["periodo"], format="%Y-%m", errors="coerce")

print("\nConversión de fechas realizada correctamente.")

# 8. RESUMEN FINAL DE EXTRACCIÓN
print("\nResumen del proceso de extracción:")
print("- Se leyó correctamente el archivo dispensaciones.csv")
print("- Se leyó correctamente el archivo compras.csv")
print("- Se leyó correctamente el archivo consumos_historicos.xlsx")
print("- Los tres datasets quedaron cargados en DataFrames de pandas")
print("- Se verificó estructura, dimensiones, tipos de datos, nulos y duplicados")

=== DISPENSACIONES ===


,id_dispensacion,fecha_dispensacion,codigo_medicamento,nombre_medicamento,presentacion,servicio_clinico,cantidad_dispensada,lote,unidad_medida
0,D0001,2025-02-01,M001,Acetaminofén 500 mg,Tableta,Urgencias,20,L202501,tabletas
1,D0002,2025-02-01,M002,Amoxicilina 500 mg,Cápsula,Consulta Externa,15,L202502,capsulas
2,D0003,2025-02-02,M003,Losartán 50 mg,Tableta,Hospitalización,30,L202503,tabletas
3,D0004,2025-02-02,M004,Insulina NPH 100 UI/mL,Frasco,UCI,5,L202504,frascos
4,D0005,2025-02-03,M005,Omeprazol 20 mg,Cápsula,Urgencias,25,L202505,capsulas


=== COMPRAS ===


,id_compra,fecha_compra,proveedor,nit_proveedor,codigo_medicamento,nombre_medicamento,cantidad_comprada,costo_unitario,lote,fecha_vencimiento
0,C0001,2025-01-20,Drogas La Rebaja,800123456,M001,Acetaminofén 500 mg,500,120.5,L202501,2027-01-15
1,C0002,2025-01-21,Copservir,900456789,M002,Amoxicilina 500 mg,300,450.0,L202502,2026-12-30
2,C0003,2025-01-22,Cruz Verde,901234567,M003,Losartán 50 mg,400,210.0,L202503,2027-02-10
3,C0004,2025-01-23,Baxter Colombia,860002222,M004,Insulina NPH 100 UI/mL,100,18500.0,L202504,2026-11-20
4,C0005,2025-01-24,Copidrogas,830111333,M005,Omeprazol 20 mg,350,310.0,L202505,2027-03-05


=== CONSUMOS HISTÓRICOS ===


,periodo,codigo_medicamento,nombre_medicamento,servicio_clinico,consumo_total,consumo_promedio_diario,unidad_medida
0,2024-11,M001,Acetaminofén 500 mg,Urgencias,620,20.7,tabletas
1,2024-11,M003,Losartán 50 mg,Hospitalización,840,28.0,tabletas
2,2024-11,M004,Insulina NPH 100 UI/mL,UCI,90,3.0,frascos
3,2024-12,M001,Acetaminofén 500 mg,Urgencias,700,22.6,tabletas
4,2024-12,M005,Omeprazol 20 mg,Urgencias,540,17.4,capsulas


Dimensiones de los datasets:
dispensaciones: 10 filas, 9 columnas
compras: 8 filas, 10 columnas
consumos_historicos: 10 filas, 7 columnas

Tipos de datos - dispensaciones
id_dispensacion        object
fecha_dispensacion     object
codigo_medicamento     object
nombre_medicamento     object
presentacion           object
servicio_clinico       object
cantidad_dispensada     int64
lote                   object
unidad_medida          object
dtype: object

Tipos de datos - compras
id_compra              object
fecha_compra           object
proveedor              object
nit_proveedor           int64
codigo_medicamento     object
nombre_medicamento     object
cantidad_comprada       int64
costo_unitario        float64
lote                   object
fecha_vencimiento      object
dtype: object

Tipos de datos - consumos_historicos
periodo                     object
codigo_medicamento          object
nombre_medicamento          object
servicio_clinico            object
consumo_total              